# 05B (Colab) — Train One Segmentation Model

Run on **Google Colab** with GPU runtime while Kaggle runs other models in parallel.

**Runtime → Change runtime type → T4 GPU**

Run once per model — change `CONFIG_NAME` each time.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
GITHUB_REPO  = 'https://github.com/yasmineelsayeddd/brain-tumor-segmentation.git'
DATASET_SLUG = 'yasmineelqorashy/brats2020-2d-prepared'
KAGGLE_TOKEN = 'KGAT_bb60250db72735c2a11893aa4a1e0db7'  # replace if expired

# Change this per run:
CONFIG_NAME    = 'unetpp.yaml'
# CONFIG_NAME  = 'attention_unet.yaml'
# CONFIG_NAME  = 'uncertainty_unet.yaml'

EPOCH_OVERRIDE = None  # set to 2 for a quick smoke test

In [ ]:
import os, shutil, subprocess, sys, json, yaml
from pathlib import Path

# Install missing packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'kaggle', 'albumentations', 'pyyaml'])

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Clone code repo ─────────────────────────────────────────────────────
PROJECT = Path('/content/project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)

subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO, str(PROJECT)])
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo cloned to:', PROJECT)

In [ ]:
# ── 2. Download prepared data from Kaggle (~12 GB, ~5 min) ─────────────────
DATA_ROOT  = Path('/content/brats2020_2d')
SPLIT_FILE = Path('/content/splits/default.json')

if not (DATA_ROOT / 'metadata.json').exists():
    print('Downloading brats2020-2d-prepared (~12 GB) …')
    dl_dir = Path('/content/kaggle_dl')
    dl_dir.mkdir(exist_ok=True)
    subprocess.check_call([
        'kaggle', 'datasets', 'download',
        '-d', DATASET_SLUG,
        '-p', str(dl_dir),
        '--unzip',
    ])
    # Dataset structure: data/brats2020_2d/ and configs/splits/default.json
    extracted = next(dl_dir.iterdir()) if len(list(dl_dir.iterdir())) == 1 and next(dl_dir.iterdir()).is_dir() else dl_dir
    src_data  = next(extracted.rglob('metadata.json')).parent
    src_split = next(extracted.rglob('default.json'))
    shutil.copytree(src_data, DATA_ROOT)
    SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_split, SPLIT_FILE)
    shutil.rmtree(dl_dir)
    print('Done.')
else:
    print('Data already present.')

with open(DATA_ROOT / 'metadata.json') as f:
    meta = json.load(f)
with open(SPLIT_FILE) as f:
    split = json.load(f)
print(f'Slices: {meta["num_slices"]:,}  |  Split: { {k: len(v) for k, v in split.items()} }')

In [ ]:
# ── 3. Build runtime config ────────────────────────────────────────────────
with open(PROJECT / 'configs' / CONFIG_NAME) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']        = str(DATA_ROOT)
cfg['data']['split_file']       = str(SPLIT_FILE)
cfg['checkpoint_dir']           = '/content/checkpoints'
cfg['experiment']['output_dir'] = '/content/outputs'

if EPOCH_OVERRIDE is not None:
    cfg['training']['epochs'] = int(EPOCH_OVERRIDE)
    cfg['training']['early_stopping_patience'] = max(2, min(int(EPOCH_OVERRIDE), 15))

runtime_cfg = Path('/content/runtime_config.yaml')
with open(runtime_cfg, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f'Model:  {cfg["model"]["arch"]}')
print(f'Epochs: {cfg["training"]["epochs"]}')
print(f'Exp:    {cfg["experiment"]["name"]}')

In [ ]:
# ── 4. Train ───────────────────────────────────────────────────────────────
run([sys.executable, '-m', 'scripts.train', '--config', str(runtime_cfg)])

In [ ]:
# ── 5. Training curves ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

exp_name  = cfg['experiment']['name']
hist_path = Path('/content/outputs') / exp_name / 'history.json'

if hist_path.exists():
    with open(hist_path) as f:
        history = json.load(f)
    epochs     = [h['epoch']      for h in history]
    train_dice = [h['train_dice'] for h in history]
    val_dice   = [h['val_dice']   for h in history]
    fig, axes  = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_dice, label='train')
    axes[0].plot(epochs, val_dice,   label='val')
    axes[0].set_title(f'{exp_name} — Dice'); axes[0].legend()
    axes[1].plot(epochs, [h['val_loss'] for h in history], color='orange')
    axes[1].set_title('Val Loss')
    plt.tight_layout(); plt.show()
    best = max(history, key=lambda h: h['val_dice'])
    print(f'Best: epoch {best["epoch"]}  val_dice={best["val_dice"]:.4f}')

In [ ]:
# ── 6. Download checkpoint to your machine ─────────────────────────────────
# Run this cell to download the .pth file before the Colab session ends
from google.colab import files

ckpt = Path('/content/checkpoints') / f'{exp_name}_best.pth'
print(f'Checkpoint: {ckpt}  exists={ckpt.exists()}')
if ckpt.exists():
    print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB')
    files.download(str(ckpt))